**Code For Rearranging order of Landmarks**

In [1]:
import os
import json
import re

# Input folder containing .mrk.json files
input_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks"

# Output folder
output_folder = os.path.join(input_folder, "sorted_mrk")
os.makedirs(output_folder, exist_ok=True)

# Desired order
desired_order = [
    "Center of femoral head",
    "Most superior point of head",
    "Most inferior point of head",
    "Neck center",
    "Tip of greater trochanter",
    "Lateral-most Tip",
    "Tip of lesser trochanter",
    "Shaft Center at Cut Plane",
    "Anterior Shaft Edge",
    "Posterior Shaft Edge",
    "Lateral Shaft Edge"
]

def normalize_label(label):

    # Remove _01, _10, -10 etc.
    label = re.sub(r'[_-]\d+$', '', label).strip()

    replacements = {
        "Center of the femoral head": "Center of femoral head",
        "Center of femoral head": "Center of femoral head",

        "Most superior point of head": "Most superior point of head",
        "Most inferior point of head": "Most inferior point of head",

        "Neck center": "Neck center",

        "Tip of greater trochanter": "Tip of greater trochanter",
        "Lateral-most Tip": "Lateral-most Tip",

        "Tip of lesser trochanter": "Tip of lesser trochanter",

        "Shaft Center at Cut Plane": "Shaft Center at Cut Plane",

        "Anterior Shaft Edge": "Anterior Shaft Edge",

        "Posterior Shaft dge": "Posterior Shaft Edge",
        "Posterior Shaft Edge": "Posterior Shaft Edge",

        "Lateral Shaft Edge": "Lateral Shaft Edge"
    }

    return replacements.get(label, label)


for filename in os.listdir(input_folder):

    if filename.endswith(".mrk.json"):

        filepath = os.path.join(input_folder, filename)

        with open(filepath, "r") as f:
            data = json.load(f)

        control_points = data["markups"][0]["controlPoints"]

        # Create dictionary
        point_dict = {}

        for point in control_points:
            label = normalize_label(point["label"])
            point_dict[label] = point

        # Check for missing landmarks
        missing = [x for x in desired_order if x not in point_dict]

        if missing:
            print(f"{filename} missing:")
            print(missing)
            continue

        # Reorder points
        new_control_points = []

        for i, name in enumerate(desired_order, start=1):
            point = point_dict[name]
            point["id"] = str(i)
            new_control_points.append(point)

        data["markups"][0]["controlPoints"] = new_control_points

        # Extract femur number from filename
        match = re.search(r'(\d+)', filename)

        if match:
            femur_num = int(match.group(1))
            output_filename = f"FemurLandmarksArranged_{femur_num:02d}.mrk.json"
        else:
            output_filename = filename

        output_path = os.path.join(output_folder, output_filename)

        with open(output_path, "w") as f:
            json.dump(data, f, indent=4)

        print(f"✓ Saved {output_filename}")

print("\nDone!")

✓ Saved FemurLandmarksArranged_01.mrk.json
✓ Saved FemurLandmarksArranged_02.mrk.json
✓ Saved FemurLandmarksArranged_03.mrk.json
✓ Saved FemurLandmarksArranged_04.mrk.json
✓ Saved FemurLandmarksArranged_05.mrk.json
✓ Saved FemurLandmarksArranged_06.mrk.json
✓ Saved FemurLandmarksArranged_07.mrk.json
✓ Saved FemurLandmarksArranged_08.mrk.json
✓ Saved FemurLandmarksArranged_09.mrk.json
✓ Saved FemurLandmarksArranged_10.mrk.json
✓ Saved FemurLandmarksArranged_11.mrk.json
✓ Saved FemurLandmarksArranged_12.mrk.json
✓ Saved FemurLandmarksArranged_13.mrk.json
✓ Saved FemurLandmarksArranged_14.mrk.json
✓ Saved FemurLandmarksArranged_15.mrk.json
✓ Saved FemurLandmarksArranged_16.mrk.json
✓ Saved FemurLandmarksArranged_17.mrk.json

Done!


**Used Rigid Transformation using Femur_10 As reference**

In [2]:
import os
import json
import numpy as np
import trimesh
from copy import deepcopy

# ==========================
# FOLDER PATH
# ==========================
base_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk"

mesh_out_folder = os.path.join(base_folder, "aligned_meshes")
landmark_out_folder = os.path.join(base_folder, "aligned_landmarks")

os.makedirs(mesh_out_folder, exist_ok=True)
os.makedirs(landmark_out_folder, exist_ok=True)

# ==========================
# REFERENCE FEMUR
# ==========================
reference_num = 10

reference_landmark_file = os.path.join(
    base_folder,
    f"FemurLandmarksArranged_{reference_num:02d}.mrk.json"
)

# ==========================
# READ LANDMARKS
# ==========================
def load_landmarks(filepath):
    with open(filepath, "r") as f:
        data = json.load(f)

    pts = []
    for cp in data["markups"][0]["controlPoints"]:
        pts.append(cp["position"])

    return np.array(pts), data


# ==========================
# KABSCH ALGORITHM
# ==========================
def rigid_transform(A, B):
    """
    Find R,t such that:

    B ≈ R*A + t

    A = moving
    B = fixed
    """

    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)

    AA = A - centroid_A
    BB = B - centroid_B

    H = AA.T @ BB

    U, S, Vt = np.linalg.svd(H)

    R = Vt.T @ U.T

    # Prevent reflection
    if np.linalg.det(R) < 0:
        Vt[2, :] *= -1
        R = Vt.T @ U.T

    t = centroid_B - R @ centroid_A

    return R, t


# ==========================
# REFERENCE LANDMARKS
# ==========================
reference_points, reference_json = load_landmarks(reference_landmark_file)

# ==========================
# PROCESS ALL FEMURS
# ==========================
for i in range(1, 18):

    mesh_file = os.path.join(base_folder, f"femur_{i:02d}.stl")
    landmark_file = os.path.join(
        base_folder,
        f"FemurLandmarksArranged_{i:02d}.mrk.json"
    )

    print(f"Processing Femur {i:02d}")

    # ----------------------
    # Reference femur
    # ----------------------
    if i == reference_num:

        # Copy mesh unchanged
        mesh = trimesh.load(mesh_file)
        mesh.export(
            os.path.join(mesh_out_folder,
                         f"Aligned_femur_{i:02d}.stl")
        )

        # Copy landmarks unchanged
        with open(landmark_file, "r") as f:
            landmark_json = json.load(f)

        with open(
            os.path.join(
                landmark_out_folder,
                f"Aligned_FemurLandmarks_{i:02d}.mrk.json"
            ),
            "w"
        ) as f:
            json.dump(landmark_json, f, indent=4)

        continue

    # ----------------------
    # Moving landmarks
    # ----------------------
    moving_points, landmark_json = load_landmarks(landmark_file)

    # Compute rigid transform
    R, t = rigid_transform(moving_points, reference_points)

    # ======================
    # Transform STL mesh
    # ======================
    mesh = trimesh.load(mesh_file)

    vertices = mesh.vertices.copy()

    vertices = (R @ vertices.T).T + t

    mesh.vertices = vertices

    mesh.export(
        os.path.join(
            mesh_out_folder,
            f"Aligned_femur_{i:02d}.stl"
        )
    )

    # ======================
    # Transform landmarks
    # ======================
    landmark_json_aligned = deepcopy(landmark_json)

    cps = landmark_json_aligned["markups"][0]["controlPoints"]

    for cp in cps:

        p = np.array(cp["position"])

        p_new = R @ p + t

        cp["position"] = p_new.tolist()

    with open(
        os.path.join(
            landmark_out_folder,
            f"Aligned_FemurLandmarks_{i:02d}.mrk.json"
        ),
        "w"
    ) as f:

        json.dump(landmark_json_aligned, f, indent=4)

print("\nDone!")

Processing Femur 01
Processing Femur 02
Processing Femur 03
Processing Femur 04
Processing Femur 05
Processing Femur 06
Processing Femur 07
Processing Femur 08
Processing Femur 09
Processing Femur 10
Processing Femur 11
Processing Femur 12
Processing Femur 13
Processing Femur 14
Processing Femur 15
Processing Femur 16
Processing Femur 17

Done!


**RBF Registration using method A**

In [101]:
import os
import json
import numpy as np
import trimesh

from scipy.interpolate import RBFInterpolator

In [ ]:
landmark_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\aligned_landmarks"

mesh_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\aligned_meshes"

output_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\rbf_registered_meshes_Method_A"

os.makedirs(output_folder, exist_ok=True)

In [109]:
def load_landmarks(filepath):

    with open(filepath, "r") as f:

        data = json.load(f)

    control_points = data["markups"][0]["controlPoints"]

    points = []

    for point in control_points:

        xyz = point["position"]

        points.append(xyz)

    points = np.array(points)

    return points

In [110]:
template_name = "16"

template_landmark_file = os.path.join(

    landmark_folder,

    "Aligned_FemurLandmarks_" + template_name + ".mrk.json"
)

template_mesh_file = os.path.join(

    mesh_folder,

    "Aligned_femur_" + template_name + ".stl"
)

template_landmarks = load_landmarks(

    template_landmark_file
)

template_mesh = trimesh.load(

    template_mesh_file
)

print(template_landmarks.shape)
print(template_mesh.vertices.shape)

(11, 3)
(24431, 3)


In [111]:
all_landmark_files = os.listdir(landmark_folder)

for landmark_file in all_landmark_files:

    if not landmark_file.endswith(".mrk.json"):

        continue


    # Skip template itself

    if landmark_file == "Aligned_FemurLandmarks_16.mrk.json":

        continue


    print()
    print("----------------------------------")
    print("Processing :", landmark_file)


    # ---------- Load source landmarks ----------

    source_landmark_path = os.path.join(

        landmark_folder,

        landmark_file
    )

    source_landmarks = load_landmarks(

        source_landmark_path
    )


    # ---------- Extract femur number ----------

    femur_number = landmark_file[-11:-9]


    # Example:
    # Aligned_FemurLandmarks_01.mrk.json
    #
    # becomes
    #
    # 01


    mesh_name = "Aligned_femur_" + femur_number + ".stl"

    source_mesh_path = os.path.join(

        mesh_folder,

        mesh_name
    )


    # ---------- Load source mesh ----------

    source_mesh = trimesh.load(

        source_mesh_path
    )

    source_vertices = source_mesh.vertices


    # ---------- Compute displacement ----------

    displacement = (

        template_landmarks

        -

        source_landmarks
    )


    # ---------- Create RBF ----------

    rbf = RBFInterpolator(

        source_landmarks,

        displacement,

        kernel="linear"
    )


    # ---------- Warp vertices ----------

    warped_vertices = (

        source_vertices

        +

        rbf(source_vertices)
    )


    # ---------- Create warped mesh ----------

    warped_mesh = trimesh.Trimesh(

        vertices=warped_vertices,

        faces=source_mesh.faces
    )


    # ---------- Save ----------

    output_name = (

        "RBF_Aligned_femur_"

        +

        femur_number

        +

        ".stl"
    )

    output_path = os.path.join(

        output_folder,

        output_name
    )


    warped_mesh.export(

        output_path
    )


    print("Saved :", output_name)

    print("Vertices :", source_vertices.shape)


----------------------------------
Processing : Aligned_FemurLandmarks_01.mrk.json
Saved : RBF_Aligned_femur_01.stl
Vertices : (22903, 3)

----------------------------------
Processing : Aligned_FemurLandmarks_02.mrk.json
Saved : RBF_Aligned_femur_02.stl
Vertices : (38588, 3)

----------------------------------
Processing : Aligned_FemurLandmarks_03.mrk.json
Saved : RBF_Aligned_femur_03.stl
Vertices : (24580, 3)

----------------------------------
Processing : Aligned_FemurLandmarks_04.mrk.json
Saved : RBF_Aligned_femur_04.stl
Vertices : (17871, 3)

----------------------------------
Processing : Aligned_FemurLandmarks_05.mrk.json
Saved : RBF_Aligned_femur_05.stl
Vertices : (25958, 3)

----------------------------------
Processing : Aligned_FemurLandmarks_06.mrk.json
Saved : RBF_Aligned_femur_06.stl
Vertices : (26975, 3)

----------------------------------
Processing : Aligned_FemurLandmarks_07.mrk.json
Saved : RBF_Aligned_femur_07.stl
Vertices : (30506, 3)

--------------------------

***Nearest-Neighbor Correspondence***

In [57]:
import os
import trimesh
import numpy as np

from scipy.spatial import cKDTree

In [62]:
template_mesh_path = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\rigid_aligned_meshes\Aligned_femur_16"

rbf_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\rbf_registered_meshes_Method_A"

output_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\correspondence_meshes"

os.makedirs(output_folder, exist_ok=True)

In [63]:
template_mesh = trimesh.load(template_mesh_path) #femur_16

template_vertices = template_mesh.vertices

template_faces = template_mesh.faces

print(template_vertices.shape)
print(template_faces.shape)

ValueError: string is not a file: `C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\rigid_aligned_meshes\Aligned_femur_16`

In [6]:
all_rbf_meshes = os.listdir(rbf_folder)

print(all_rbf_meshes)

['RBF_Aligned_femur_01.stl', 'RBF_Aligned_femur_02.stl', 'RBF_Aligned_femur_03.stl', 'RBF_Aligned_femur_04.stl', 'RBF_Aligned_femur_05.stl', 'RBF_Aligned_femur_06.stl', 'RBF_Aligned_femur_07.stl', 'RBF_Aligned_femur_08.stl', 'RBF_Aligned_femur_09.stl', 'RBF_Aligned_femur_10.stl', 'RBF_Aligned_femur_11.stl', 'RBF_Aligned_femur_12.stl', 'RBF_Aligned_femur_13.stl', 'RBF_Aligned_femur_14.stl', 'RBF_Aligned_femur_15.stl', 'RBF_Aligned_femur_17.stl']


In [56]:
for mesh_file in all_rbf_meshes:

    if not mesh_file.endswith(".stl"):
        continue

    print()
    print("Processing :", mesh_file)


    # ------------------------
    # Load RBF mesh
    # ------------------------

    mesh_path = os.path.join(
        rbf_folder,
        mesh_file
    )

    warped_mesh = trimesh.load(
        mesh_path
    )

    warped_vertices = warped_mesh.vertices


    # ------------------------
    # Build KD Tree
    # ------------------------

    tree = cKDTree(
        warped_vertices
    )


    # ------------------------
    # Find nearest neighbor
    # for every template vertex
    # ------------------------

    distances, indices = tree.query(
        template_vertices
    )


    # ------------------------
    # Get corresponding points
    # ------------------------

    new_vertices = warped_vertices[
        indices
    ]


    # ------------------------
    # Create mesh
    # ------------------------

    new_mesh = trimesh.Trimesh(

        vertices=new_vertices,

        faces=template_faces
    )


    # ------------------------
    # Save
    # ------------------------

    output_name = "Correspondence_" + mesh_file

    output_path = os.path.join(
        output_folder,
        output_name
    )

    new_mesh.export(
        output_path
    )


    print("Saved :", output_name)
    print(warped_mesh.vertices)
    print(new_vertices.shape)


Processing : RBF_Aligned_femur_01.stl


ValueError: string is not a file: `C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\rbf_registered_meshes\RBF_Aligned_femur_01.stl`

In [55]:
mesh = trimesh.load(
    r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\correspondence_meshes\Correspondence_RBF_Aligned_femur_14.stl"
)

print(mesh.vertices.shape)

(20027, 3)


**Method B**

In [34]:
import os
import json
import numpy as np
import trimesh

from scipy.interpolate import RBFInterpolator

In [35]:
landmark_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\aligned_landmarks"

mesh_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\aligned_meshes"

output_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\rbf_registered_meshes_Method_B"

os.makedirs(output_folder, exist_ok=True)

In [36]:
def load_landmarks(filepath):

    with open(filepath, "r") as f:
        data = json.load(f)

    control_points = data["markups"][0]["controlPoints"]

    points = []

    for point in control_points:

        points.append(point["position"])

    return np.array(points)

In [37]:
template_name = "16"

template_mesh_path = os.path.join(
    mesh_folder,
    "Aligned_femur_" + template_name + ".stl"
)

template_landmark_path = os.path.join(
    landmark_folder,
    "Aligned_FemurLandmarks_" + template_name + ".mrk.json"
)

template_mesh = trimesh.load(template_mesh_path)

template_vertices = template_mesh.vertices

template_faces = template_mesh.faces

template_landmarks = load_landmarks(
    template_landmark_path
)

print(template_vertices.shape)
print(template_landmarks.shape)

(24431, 3)
(11, 3)


In [38]:
all_landmark_files = os.listdir(
    landmark_folder
)

for landmark_file in all_landmark_files:

    if not landmark_file.endswith(".mrk.json"):
        continue

    if landmark_file == "Aligned_FemurLandmarks_16.mrk.json":
        continue


    print()
    print("Processing :", landmark_file)


    # --------------------------------
    # Extract femur number
    # --------------------------------

    femur_number = landmark_file[-11:-9]


    # --------------------------------
    # Load subject landmarks
    # --------------------------------

    subject_landmark_path = os.path.join(
        landmark_folder,
        landmark_file
    )

    subject_landmarks = load_landmarks(
        subject_landmark_path
    )


    # --------------------------------
    # Landmark displacement
    # --------------------------------

    displacement = (
        subject_landmarks
        -
        template_landmarks
    )


    # --------------------------------
    # Build RBF
    # --------------------------------

    rbf = RBFInterpolator(

        template_landmarks,

        displacement,

        kernel="linear"
    )


    # --------------------------------
    # Warp TEMPLATE vertices
    # --------------------------------

    warped_vertices = (
        template_vertices
        +
        rbf(template_vertices)
    )


    # --------------------------------
    # Create mesh
    # --------------------------------

    warped_mesh = trimesh.Trimesh(

        vertices=warped_vertices,

        faces=template_faces
    )


    # --------------------------------
    # Save
    # --------------------------------

    output_name = (
        "Warped_femur_"
        +
        femur_number
        +
        ".stl"
    )

    output_path = os.path.join(
        output_folder,
        output_name
    )

    warped_mesh.export(
        output_path
    )

    print("Saved :", output_name)


Processing : Aligned_FemurLandmarks_01.mrk.json
Saved : Warped_femur_01.stl

Processing : Aligned_FemurLandmarks_02.mrk.json
Saved : Warped_femur_02.stl

Processing : Aligned_FemurLandmarks_03.mrk.json
Saved : Warped_femur_03.stl

Processing : Aligned_FemurLandmarks_04.mrk.json
Saved : Warped_femur_04.stl

Processing : Aligned_FemurLandmarks_05.mrk.json
Saved : Warped_femur_05.stl

Processing : Aligned_FemurLandmarks_06.mrk.json
Saved : Warped_femur_06.stl

Processing : Aligned_FemurLandmarks_07.mrk.json
Saved : Warped_femur_07.stl

Processing : Aligned_FemurLandmarks_08.mrk.json
Saved : Warped_femur_08.stl

Processing : Aligned_FemurLandmarks_09.mrk.json
Saved : Warped_femur_09.stl

Processing : Aligned_FemurLandmarks_10.mrk.json
Saved : Warped_femur_10.stl

Processing : Aligned_FemurLandmarks_11.mrk.json
Saved : Warped_femur_11.stl

Processing : Aligned_FemurLandmarks_12.mrk.json
Saved : Warped_femur_12.stl

Processing : Aligned_FemurLandmarks_13.mrk.json
Saved : Warped_femur_13.stl

In [39]:
template_vertices.shape

(24431, 3)

In [66]:
mesh = trimesh.load(
    r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\rbf_registered_meshes_Method_B\Warped_femur_01.stl"
)

print(mesh.vertices.shape)
print(mesh.faces.shape)

(24431, 3)
(48718, 3)


**PCA**

In [1]:
import os
import trimesh
import numpy as np

In [2]:
mesh_folder = r"C:\Users\Yash\OneDrive\Desktop\Nit Internship\Base paper Implementation\Dataset\Experimental Data\11 landmarks\sorted_mrk\rbf_registered_meshes_Method_B"

In [3]:
shape_matrix = []

all_meshes = os.listdir(mesh_folder)

for file in all_meshes:

    if not file.endswith(".stl"):
        continue

    mesh_path = os.path.join(
        mesh_folder,
        file
    )

    mesh = trimesh.load(mesh_path)

    vertices = mesh.vertices

    vector = vertices.flatten()

    shape_matrix.append(vector)

In [4]:
shape_matrix = np.array(shape_matrix)

print(shape_matrix.shape)

(16, 73293)


In [5]:
shape_matrix

array([[334.82455444, 167.78796387, 758.18945312, ..., 341.13586426,
        190.22581482, 922.27539062],
       [338.60253906, 163.95199585, 733.73077393, ..., 337.31137085,
        205.68974304, 967.03631592],
       [341.70532227, 165.11697388, 761.72955322, ..., 341.38787842,
        186.46936035, 916.74194336],
       ...,
       [327.89208984, 169.09066772, 750.93548584, ..., 341.63247681,
        195.66087341, 937.97235107],
       [334.65826416, 152.42785645, 732.81652832, ..., 340.79272461,
        202.88729858, 964.47027588],
       [330.1882019 , 159.56985474, 742.41278076, ..., 348.2170105 ,
        200.25343323, 952.67474365]], shape=(16, 73293))

In [6]:
mean_shape = np.mean(
    shape_matrix,
    axis=0
)

print(mean_shape.shape)

(73293,)


In [7]:
shape_matrix

array([[334.82455444, 167.78796387, 758.18945312, ..., 341.13586426,
        190.22581482, 922.27539062],
       [338.60253906, 163.95199585, 733.73077393, ..., 337.31137085,
        205.68974304, 967.03631592],
       [341.70532227, 165.11697388, 761.72955322, ..., 341.38787842,
        186.46936035, 916.74194336],
       ...,
       [327.89208984, 169.09066772, 750.93548584, ..., 341.63247681,
        195.66087341, 937.97235107],
       [334.65826416, 152.42785645, 732.81652832, ..., 340.79272461,
        202.88729858, 964.47027588],
       [330.1882019 , 159.56985474, 742.41278076, ..., 348.2170105 ,
        200.25343323, 952.67474365]], shape=(16, 73293))

In [ ]:
mean_shape  

array([333.66304398, 161.47506332, 748.53108215, ..., 340.29153442,
       197.25537872, 939.13976669], shape=(73293,))

In [9]:
centered_data = (
    shape_matrix
    -
    mean_shape
)

In [10]:
centered_data

array([[  1.16151047,   6.31290054,   9.65837097, ...,   0.84432983,
         -7.0295639 , -16.86437607],
       [  4.93949509,   2.47693253, -14.80030823, ...,  -2.98016357,
          8.43436432,  27.89654922],
       [  8.04227829,   3.64191055,  13.19847107, ...,   1.09634399,
        -10.78601837, -22.39782333],
       ...,
       [ -5.77095413,   7.6156044 ,   2.40440369, ...,   1.34094238,
         -1.59450531,  -1.16741562],
       [  0.99522018,  -9.04720688, -15.71455383, ...,   0.50119019,
          5.63191986,  25.33050919],
       [ -3.47484207,  -1.90520859,  -6.11830139, ...,   7.92547607,
          2.9980545 ,  13.53497696]], shape=(16, 73293))

In [11]:
from sklearn.decomposition import PCA

pca = PCA()

pca.fit(centered_data)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",None
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized S

**Cumulative explained variance ratio**
(#How much of the total shape variation have I captured?)

In [12]:
variance_ratio = pca.explained_variance_ratio_

cumulative_variance = np.cumsum(
    variance_ratio
)

print(cumulative_variance)

[0.89059706 0.93329187 0.95777192 0.97361023 0.9817853  0.9866658
 0.99060634 0.99390938 0.99621035 0.99799623 0.99882935 0.99947976
 0.99983553 0.99994299 1.         1.        ]


In [13]:
print(shape_matrix.shape)
print(centered_data.shape)

(16, 73293)
(16, 73293)


**(Eigen Vectors)**

In [39]:
print(pca.components_)

[[-4.26621834e-04 -2.40113998e-03 -6.81031409e-03 ... -4.28726774e-04
   2.28110400e-03  1.08511312e-02]
 [ 3.33708168e-03 -8.32499186e-03 -3.91668340e-03 ... -3.36040333e-03
  -7.00324149e-04  1.05162532e-03]
 [ 7.25902384e-03  4.01192488e-03 -4.56555999e-03 ... -1.79842973e-03
  -1.22061121e-02 -1.04283243e-03]
 ...
 [-4.51159164e-03  2.38215421e-04 -8.09552772e-03 ... -7.89982564e-03
   5.61988570e-03 -9.32966804e-03]
 [ 5.81927899e-03  6.07949944e-03  3.52407555e-04 ... -3.35527018e-03
  -8.51657710e-03 -1.70997292e-02]
 [ 9.93265130e-01 -1.40804870e-02  1.02141451e-02 ... -9.91126430e-05
   2.92996167e-05 -8.13048990e-05]]


**Eigenvalues That Help Compute σ and normalize scores (lambda values)**

In [15]:
print(pca.explained_variance_)

[3.43562629e+06 1.64702305e+05 9.44358635e+04 6.10989462e+04
 3.15366635e+04 1.88273671e+04 1.52012633e+04 1.27420251e+04
 8.87639761e+03 6.88930507e+03 3.21389253e+03 2.50909733e+03
 1.37241990e+03 4.14564948e+02 2.19913246e+02 1.11878746e-23]


**Finding p-values for all the femurs in the dataset using PCA**

In [21]:
scores=pca.transform(centered_data)
scores

array([[-1.55961909e+03, -2.98426139e+02,  4.02701781e+02,
         2.72640587e+01, -1.09525865e+02, -4.20796622e+01,
        -1.09253457e+02, -5.62885222e+01,  9.13594109e+01,
        -1.73772430e+02,  1.89245362e+01,  8.58526302e+01,
         2.28245045e+00, -1.31264207e+01, -1.31321024e+01,
         5.03037126e-11],
       [ 2.59287929e+03, -4.65350520e+02,  3.19055505e+02,
         2.36585709e+02,  8.19976751e+01, -8.92931564e+01,
         1.01057646e+02,  1.26185875e+02,  1.95558503e+02,
         7.17354053e+01, -1.19984528e+01, -8.49554200e+00,
        -3.16432141e+01,  1.23442148e+01,  1.66318712e+00,
        -4.90448480e-12],
       [-2.11587417e+03,  2.30039489e+02,  2.57620374e+02,
         3.40689175e+01,  5.83697491e+01, -1.34341247e+02,
        -1.54406613e+02,  8.34782475e+01, -1.63782884e+01,
         1.14719884e+02,  1.04277173e+02, -5.15650493e+01,
         4.01465117e+01,  4.45391783e+00, -1.42902829e+01,
        -3.52472218e-12],
       [-2.58454480e+03, -3.92396328e

In [20]:
scores.shape

(16, 16)

**Normalizing p-values**

In [ ]:
scores_5 = scores[:,:5]

std = np.sqrt(pca.explained_variance_[:5]) 

normalized_scores_5 = scores_5/std #pi​=​scorei/sqt(λi​)​​

In [32]:
normalized_scores_5

array([[-0.84142533, -0.73533802,  1.31043376,  0.11029953, -0.61674952],
       [ 1.39887639, -1.14664865,  1.03824002,  0.95713158,  0.46173593],
       [-1.14152881,  0.56682964,  0.83832367,  0.13782928,  0.32868505],
       [-1.39437987, -0.00966886, -1.8653579 ,  2.33983764, -1.50500161],
       [-0.25618331, -0.20678039,  1.23310658, -0.56611615, -0.45017984],
       [ 0.39519599,  1.15565284,  0.48825312, -0.70912854, -0.80587962],
       [-0.68309768,  1.03682647,  0.39221809,  0.57165357,  1.18709769],
       [ 0.2174226 ,  0.81541532, -0.94749717, -0.05063354,  0.89606592],
       [-1.23858997, -0.01048144,  0.26090555, -0.47677694,  0.8413703 ],
       [ 1.5616445 ,  1.46072826,  0.81347816,  0.46844034, -1.72155305],
       [ 0.70665389, -0.48371963, -0.55417862,  0.6363407 ,  1.99240017],
       [ 0.54060512,  0.71939823, -1.8906439 , -2.03765703, -0.24085849],
       [-1.22242513, -0.05793998,  0.16407015, -1.06788151,  0.15335973],
       [-0.01641986, -1.61507189, -0.4

**Creating a csv file containing p-values for all the femurs in the dataset**

In [36]:
import pandas as pd

df = pd.DataFrame(
{
    "image_name":[
        "Femur_01.png",
        "Femur_02.png",
        "Femur_03.png",
        "Femur_04.png",
        "Femur_05.png",
        "Femur_06.png",
        "Femur_07.png",
        "Femur_08.png",
        "Femur_09.png",
        "Femur_10.png",
        "Femur_11.png",
        "Femur_12.png",
        "Femur_13.png",
        "Femur_14.png",
        "Femur_15.png",
        "Femur_17.png"
    ],

    "p1":normalized_scores_5[:,0],
    "p2":normalized_scores_5[:,1],
    "p3":normalized_scores_5[:,2],
    "p4":normalized_scores_5[:,3],
    "p5":normalized_scores_5[:,4]
}
)

df.to_csv(
    "ground_truth_p-values_labels.csv",
    index=False
)

In [37]:
df=pd.read_csv('ground_truth_p-values_labels.csv')
print(df)

      image_name        p1        p2        p3        p4        p5
0   Femur_01.png -0.841425 -0.735338  1.310434  0.110300 -0.616750
1   Femur_02.png  1.398876 -1.146649  1.038240  0.957132  0.461736
2   Femur_03.png -1.141529  0.566830  0.838324  0.137829  0.328685
3   Femur_04.png -1.394380 -0.009669 -1.865358  2.339838 -1.505002
4   Femur_05.png -0.256183 -0.206780  1.233107 -0.566116 -0.450180
5   Femur_06.png  0.395196  1.155653  0.488253 -0.709129 -0.805880
6   Femur_07.png -0.683098  1.036826  0.392218  0.571654  1.187098
7   Femur_08.png  0.217423  0.815415 -0.947497 -0.050634  0.896066
8   Femur_09.png -1.238590 -0.010481  0.260906 -0.476777  0.841370
9   Femur_10.png  1.561645  1.460728  0.813478  0.468440 -1.721553
10  Femur_11.png  0.706654 -0.483720 -0.554179  0.636341  1.992400
11  Femur_12.png  0.540605  0.719398 -1.890644 -2.037657 -0.240858
12  Femur_13.png -1.222425 -0.057940  0.164070 -1.067882  0.153360
13  Femur_14.png -0.016420 -1.615072 -0.475570 -0.733156 -0.79